In [1]:
import pandas as pd
import os
import glob

In [2]:
def parse_rafdb_rows(folder_path):
    if not os.path.exists(folder_path):
        print(f"Error: Folder not found at {folder_path}")
        return

    txt_files = glob.glob(os.path.join(folder_path, '*_attri.txt'))
    
    if len(txt_files) == 0:
        print("No attribute files found.")
        return

    print(f"Found {len(txt_files)} files. Extracting Row 6 (Gender) and Row 7 (Race)")

    data = []
    
    for file_path in txt_files:
        try:
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            if len(lines) >= 7:
                gender_code = int(lines[5].strip()) 
                race_code = int(lines[6].strip())   
                
                if gender_code in [0, 1] and race_code in [0, 1, 2]:
                    data.append({
                        'Filename': os.path.basename(file_path),
                        'Gender_Code': gender_code,
                        'Race_Code': race_code
                    })
            else:
                pass
                
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    if not data:
        print("No valid data found.")
        return

    df = pd.DataFrame(data)
    
    race_map = {0: 'Caucasian', 1: 'African-American', 2: 'Asian'}
    gender_map = {0: 'Male', 1: 'Female'}
    
    df['Race'] = df['Race_Code'].map(race_map)
    df['Gender'] = df['Gender_Code'].map(gender_map)
    
    table = pd.crosstab(df['Race'], df['Gender'], margins=True, margins_name="Total")
    
    print("\n=== Demographic Distribution (Count) ===")
    print(table)
    
    grand_total = table.loc['Total', 'Total']
    table_percent = (table / grand_total * 100).round(1).astype(str) + '%'
    
    print("\n=== Demographic Distribution (Percentage) ===")
    print(table_percent)

In [3]:
ANNOTATION_FOLDER = r'../RafDB/basic/Annotation/manual'
parse_rafdb_rows(ANNOTATION_FOLDER)

Found 15339 files. Extracting Row 6 (Gender) and Row 7 (Race)

=== Demographic Distribution (Count) ===
Gender            Female  Male  Total
Race                                 
African-American     476   610   1086
Asian               1349   869   2218
Caucasian           6357  4727  11084
Total               8182  6206  14388

=== Demographic Distribution (Percentage) ===
Gender           Female   Male   Total
Race                                  
African-American   3.3%   4.2%    7.5%
Asian              9.4%   6.0%   15.4%
Caucasian         44.2%  32.9%   77.0%
Total             56.9%  43.1%  100.0%


In [4]:
def calculate_clean_split_counts(folder_path):
    if not os.path.exists(folder_path):
        print(f"Error: Folder not found at {folder_path}")
        return

    print(f"Scanning {folder_path}...")
    txt_files = glob.glob(os.path.join(folder_path, '*_attri.txt'))
    
    if len(txt_files) == 0:
        print("No attribute files found.")
        return

    train_count = 0
    test_count = 0
    skipped_unsure = 0
    
    print(f"Processing {len(txt_files)} files...")
    
    for file_path in txt_files:
        try:
            filename = os.path.basename(file_path).lower()
            
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            if len(lines) >= 6:
                gender_code = int(lines[5].strip())
                
                if gender_code in [0, 1]:
                    if 'train' in filename:
                        train_count += 1
                    elif 'test' in filename:
                        test_count += 1
                else:
                    skipped_unsure += 1
            else:
                pass
                
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    total_clean = train_count + test_count
    
    print("\n" + "="*40)
    print("      FINAL SAMPLE COUNTS (Gender Filtered)")
    print("="*40)
    print(f"Train Samples (Clean):  {train_count}")
    print(f"Test Samples (Clean):   {test_count}")
    print("-" * 40)
    print(f"Total Samples (Clean):  {total_clean}")
    print(f"Removed (Unsure/Other): {skipped_unsure}")
    print("="*40)

In [5]:
calculate_clean_split_counts(ANNOTATION_FOLDER)

Scanning ../RafDB/basic/Annotation/manual...
Processing 15339 files...

      FINAL SAMPLE COUNTS (Gender Filtered)
Train Samples (Clean):  11519
Test Samples (Clean):   2869
----------------------------------------
Total Samples (Clean):  14388
Removed (Unsure/Other): 951
